In [1]:
%load_ext autoreload
%autoreload 2

In [295]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

In [296]:
SEED = 1

CART_POSITION_RANGE = 20 # -4.8 -2.4 ... 2.4 4.8
CART_VELOCITY_RANGE = 20 # -inf -2 ... 2 inf
POLE_POSITION_RANGE = 20 # -0.418 -0.2095 ... -0.2095 0.418
POLE_VELOCITY_RANGE = 20 # -inf -2 ... 2 inf

In [297]:
def make_environment() -> tuple[list[float], list[float], list[float], list[float]]:
	cart_pos: list[float] = []
	cart_velo: list[float] = []
	pole_pos: list[float] = []
	pole_velo: list[float] = []
	
	cart_pos.append(-4.8)
	cart_pos.append(-2.4)
	cart_coef = 4.8 / CART_POSITION_RANGE
	for i in range(1, CART_POSITION_RANGE):
		cart_pos.append(-2.4 + cart_coef * i)
	cart_pos.append(2.4)
	cart_pos.append(4.8)
	print(cart_pos)

	cart_velo.append(float("-inf"))
	cart_velo.append(-2)
	cart_coef = 4 / CART_VELOCITY_RANGE
	for i in range(1, CART_VELOCITY_RANGE):
		cart_velo.append(-2 + cart_coef * i)
	cart_velo.append(2)
	cart_velo.append(float("inf"))
	print(cart_velo)

	pole_pos.append(-0.418)
	pole_pos.append(-0.2095)
	cart_coef = 0.418 / POLE_POSITION_RANGE
	for i in range(1, POLE_POSITION_RANGE):
		pole_pos.append(-0.2095 + cart_coef * i)
	pole_pos.append(0.2095)
	pole_pos.append(0.418)
	print(pole_pos)

	pole_velo.append(float("-inf"))
	pole_velo.append(-2)
	cart_coef = 4 / POLE_VELOCITY_RANGE
	for i in range(1, POLE_VELOCITY_RANGE):
		pole_velo.append(-2 + cart_coef * i)
	pole_velo.append(2)
	pole_velo.append(float("inf"))
	print(pole_velo)
	
	return cart_pos, cart_velo, pole_pos, pole_velo

CART_POS, CART_VELO, POLE_POS, POLE_VELO = make_environment()
print(len(CART_POS))
print(len(CART_VELO))
print(len(POLE_POS))
print(len(POLE_VELO))
MAX_STATE = 0
MAX_STATE = MAX_STATE * len(CART_POS) + len(CART_POS) - 2
MAX_STATE = MAX_STATE * len(CART_VELO) + len(CART_VELO) - 2
MAX_STATE = MAX_STATE * len(POLE_POS) + len(POLE_POS) - 2
MAX_STATE = MAX_STATE * len(POLE_VELO) + len(POLE_VELO) - 2
print(MAX_STATE)

[-4.8, -2.4, -2.16, -1.92, -1.68, -1.44, -1.2, -0.96, -0.72, -0.48, -0.23999999999999977, 0.0, 0.23999999999999977, 0.48, 0.7200000000000002, 0.96, 1.1999999999999997, 1.44, 1.6800000000000002, 1.9200000000000004, 2.1599999999999997, 2.4, 4.8]
[-inf, -2, -1.8, -1.6, -1.4, -1.2, -1.0, -0.7999999999999998, -0.5999999999999999, -0.3999999999999999, -0.19999999999999996, 0.0, 0.20000000000000018, 0.40000000000000036, 0.6000000000000001, 0.8000000000000003, 1.0, 1.2000000000000002, 1.4000000000000004, 1.6, 1.8000000000000003, 2, inf]
[-0.418, -0.2095, -0.1886, -0.1677, -0.14679999999999999, -0.1259, -0.105, -0.08410000000000001, -0.0632, -0.042300000000000004, -0.021400000000000002, -0.0005000000000000004, 0.0204, 0.041299999999999976, 0.062200000000000005, 0.08309999999999998, 0.10400000000000001, 0.12489999999999998, 0.14579999999999996, 0.1667, 0.18759999999999996, 0.2095, 0.418]
[-inf, -2, -1.8, -1.6, -1.4, -1.2, -1.0, -0.7999999999999998, -0.5999999999999999, -0.3999999999999999, -0.19

In [298]:
def get_state(obs) -> int:
	cart_pos = len(CART_POS)
	for i in range(len(CART_POS) - 1):
		cart_pos = i
		if obs[0] <= CART_POS[i + 1]:
			break
	
	cart_velo = len(CART_VELO)
	for i in range(len(CART_VELO) - 1):
		cart_velo = i
		if obs[1] <= CART_VELO[i + 1]:
			break
	
	pole_pos = len(POLE_POS)
	for i in range(len(POLE_POS) - 1):
		pole_pos = i
		if obs[2] <= POLE_POS[i + 1]:
			break
	
	pole_velo = len(POLE_VELO)
	for i in range(len(POLE_VELO) - 1):
		pole_velo = i
		if obs[3] <= POLE_VELO[i + 1]:
			break
	state = 0
	state = state * len(CART_POS) + cart_pos
	state = state * len(CART_VELO) + cart_velo
	state = state * len(POLE_POS) + pole_pos
	state = state * len(POLE_VELO) + pole_velo
	return state

In [299]:
# env = gym.make("CartPole-v1", sutton_barto_reward=True)
env = gym.make("CartPole-v1")
env.action_space.seed(SEED)
env.reset(seed=SEED)

print(env.action_space)
print(env.observation_space)
obs = env.observation_space.sample()
print(obs)
print(get_state([-4.8, float("-inf"), -0.418, float("-inf")]))
print(get_state([4.8, float("inf"), 0.418, float("inf")]))

Discrete(2)
Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
[-3.0503685   0.24331145  0.20082586  1.5276655 ]
0
267120


# Simulate lại mô hình xác suất

In [300]:
sas_probabilistic: dict[tuple[int, int, int], float] = defaultdict(float)
sa_sum: dict[tuple[int, int], int] = defaultdict(int)
is_end: dict[int, bool] = defaultdict(bool)

SIMULATED_EPISODES = 50_000
for i in range(SIMULATED_EPISODES):
	obs, info = env.reset()
	# print(f"Starting obs: {obs}")
	episode_over = False
	total_reward = 0
	while not episode_over:
		action = env.action_space.sample()
		nxt_obs, reward, terminated, truncated, info = env.step(action)
		sa_sum[get_state(obs), action] += 1
		sas_probabilistic[get_state(obs), action, get_state(nxt_obs)] += 1
		
		obs = nxt_obs
		episode_over = terminated or truncated
	is_end[get_state(obs)] = True

sa_nxt_state: dict[tuple[int, int], list[int]] = defaultdict(lambda: [])
for obs, action, nxt_obs in sas_probabilistic:
	print(f"Before ({obs}, {action}, {nxt_obs}): {sas_probabilistic[obs, action, nxt_obs]}")
	sas_probabilistic[obs, action, nxt_obs] /= sa_sum[obs, action]
	sa_nxt_state[obs, action].append(nxt_obs)
	print(f"After ({obs}, {action}, {nxt_obs}): {sas_probabilistic[obs, action, nxt_obs]}")
print(f"Finish simulated")

Before (127246, 0, 126719): 1178.0
After (127246, 0, 126719): 0.8725925925925926
Before (126719, 1, 127247): 1364.0
After (126719, 1, 127247): 0.48976660682226214
Before (127247, 1, 127774): 2176.0
After (127247, 1, 127774): 0.7835794022326251
Before (127774, 1, 128302): 1375.0
After (127774, 1, 128302): 0.6213285133303208
Before (128302, 0, 127774): 307.0
After (128302, 0, 127774): 0.24758064516129033
Before (127774, 0, 127224): 396.0
After (127774, 0, 127224): 0.17861975642760486
Before (127224, 1, 127751): 1870.0
After (127224, 1, 127751): 0.8091735179575941
Before (127751, 1, 128279): 1363.0
After (127751, 1, 128279): 0.5473895582329317
Before (128279, 0, 139918): 51.0
After (128279, 0, 139918): 0.039504260263361735
Before (139918, 0, 139368): 381.0
After (139918, 0, 139368): 0.16464995678478825
Before (139368, 1, 139918): 58.0
After (139368, 1, 139918): 0.029546612328069283
Before (139368, 0, 138840): 1498.0
After (139368, 0, 138840): 0.7460159362549801
Before (138840, 1, 139391):

In [301]:
# def run_under_policy(policy: list[int], gamma: float, num_runs: int) -> tuple[float, int]:
def run_under_policy(policy: list[int], gamma: float, num_runs: int) -> float:
	env.action_space.seed(1)
	avg_rewards = 0
	avg_lengths = 0
	for i in range(num_runs):
		obs, info = env.reset(seed=1 if i == 0 else None)
		episode_over = False
		# rewards = []
		rewards = 0
		while not episode_over:
			action = policy[get_state(obs)]
			nxt_obs, reward, terminated, truncated, info = env.step(action)
			# rewards.append(reward)
			rewards += reward
			obs = nxt_obs
			episode_over = terminated or truncated
		# reward = sum(rewards[i] * (gamma ** i) for i in range(len(rewards)))
		# avg_lengths += len(rewards)
		avg_rewards += rewards
	avg_rewards /= num_runs
	# avg_lengths /= num_runs
	# return avg_rewards, avg_lengths
	return avg_rewards

# Policy iteration

In [ ]:
from tqdm import tqdm

def policy_iteration(gamma: float) -> tuple[list[list[int]], list[int]]:
	q_values: list[list[int]] = [[0, 0] for i in range(0, MAX_STATE + 1)]
	pi: list[int] = [0 if sa_sum[i, 0] > 0 else 1 for i in range(0, MAX_STATE + 1)]
	print(len(q_values))
	print(len(pi))
	MAX_TOLERANCE = 1e-4

	for state in range(0, MAX_STATE + 1):
		if is_end[state]:
			continue
		for action in range(2):
			if (sa_sum[state, action] == 0):
				q_values[state][action] = float("-inf")

	n_episodes = 100_000
	progress_bar = tqdm(range(n_episodes))
	for episode in progress_bar:
		while True:
			tolerance = 0
			for state in range(0, MAX_STATE + 1):
				if is_end[state]:
					continue
				for action in range(2):
					if (sa_sum[state, action] == 0):
						continue
					old_value = q_values[state][action]
					new_value = 0
					for nxt_state in sa_nxt_state[state, action]:
						reward = 1 if not is_end[nxt_state] else 0
						new_value += sas_probabilistic[state, action, nxt_state] * (reward + gamma * q_values[nxt_state][pi[nxt_state]])
					q_values[state][action] = new_value
					tolerance = max(tolerance, abs(new_value - old_value))
			if tolerance < MAX_TOLERANCE:
				break
			# print(tolerance)
		policy_stable = True

		for state in range(0, MAX_STATE + 1):
			if is_end[state]:
				continue
			old_action = pi[state]
			new_action = np.argmax(q_values[state])
			if old_action != new_action:
				policy_stable = False
			pi[state] = new_action
		if policy_stable:
			break
		# avg_rewards, avg_lengths = run_under_policy(pi, gamma, 50)
		# progress_bar.set_postfix(avg_rewards=avg_rewards, avg_lengths=avg_lengths)
		avg_rewards = run_under_policy(pi, gamma, 500)
		progress_bar.set_postfix(avg_rewards=avg_rewards)

	return q_values, pi

In [332]:
q_values, pi = policy_iteration(gamma=0.95)

267121
267121


  0%|          | 30/100000 [06:23<355:26:47, 12.80s/it, avg_rewards=106] 


In [275]:
env = gym.make('CartPole-v1', render_mode="human")
# env.action_space.seed(1)
# observation, info = env.reset(seed=1)
for i in range(5):
	obs, info = env.reset(seed=1 if i == 0 else None)
	print(f"Starting observation: {obs}")
	episode_over = False
	total_reward = 0
	while not episode_over:
		action = pi[get_state(obs)]
		nxt_obs, reward, terminated, truncated, info = env.step(action)
		obs = nxt_obs
		total_reward += 1
		episode_over = terminated or truncated
	print(f"Episode finished! Total reward: {total_reward}")
env.close()

Starting observation: [ 0.00118216  0.04504637 -0.03558404  0.04486495]
Episode finished! Total reward: 10
Starting observation: [-0.01881685 -0.00766736  0.03277026 -0.00908009]
Episode finished! Total reward: 9
Starting observation: [ 0.00495937 -0.04724409  0.02535131  0.00381433]
Episode finished! Total reward: 9
Starting observation: [-0.01702683  0.02884287 -0.01968052 -0.00465021]
Episode finished! Total reward: 10
Starting observation: [-0.03659583 -0.0096887  -0.02965448 -0.02376867]
Episode finished! Total reward: 10
